*Nama = Ivana Tiara Rante*

*NIM = H061251058*

# Promt 1
## Produk yang dapat dihasilkan dari tugas 1

## Analisis data geolistrik resistivitas

Analisis Awal Resistivitas Bawah Permukaan

Metode: Geolistrik resistivitas

Tujuan: Menghitung resistivitas semu dan melihat variasi nilai resistivitas berdasarkan data pengukuran.

### 1. Input data contoh

Karena file tugasmu belum menyertakan data pengukuran geolistrik, tabel berikut menggunakan data ilustrasi untuk demonstrasi, bukan hasil survei lapangan.

| Titik | Arus \\(I\\) (A) | Beda potensial \\(\Delta V\\) (V) | Faktor geometri \\(K\\) (m) |
| ----- | ---------------- | --------------------------------- | --------------------------- |
| 1     | 0,10             | 0,05                              | 20                          |
| 2     | 0,10             | 0,10                              | 20                          |
| 3     | 0,10             | 0,15                              | 20                          |

### 2. Proses perhitungan

Rumus resistivitas semu:

\\[ \rho_a=K\frac{\Delta V}{I} \\]

Contoh perhitungan titik pertama:

\\[ \rho_a=20\times\frac{0{,}05}{0{,}10} \\]

\\[ \boxed{\rho_a=10\ \Omega\text{m}} \\]

### 3. Output produk

| Titik | Resistivitas semu (\\(\Omega\text{m}\\)) |
| ----- | ---------------------------------------- |
| 1     | 10                                       |
| 2     | 20                                       |
| 3     | 30                                       |

Hasil Perhitungan Resistivitas Semu

Nilai ilustrasi dari tiga titik pengukuran.

Titik 1

Resistivitas semu10 Ωm

0 Ωm8 Ωm16 Ωm24 Ωm32 ΩmTitik 1Titik 2Titik 3

### 4. Interpretasi hasil

Berdasarkan data ilustrasi, nilai resistivitas semu meningkat dari 10 \\(\Omega\text{m}\\) pada titik pertama menjadi 30 \\(\Omega\text{m}\\) pada titik ketiga. Perubahan ini menunjukkan adanya variasi respons kelistrikan bawah permukaan pada titik pengukuran tersebut.

Namun, jenis tanah atau batuan belum dapat ditentukan secara pasti hanya dari nilai tersebut. Interpretasi lebih lanjut memerlukan informasi konfigurasi elektroda, jarak pengukuran, dan proses inversi untuk memperoleh model bawah permukaan.



 # Prompt 2
 ## Analisis Seismik Refraksi


In [29]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# DATA SIMULASI untuk model ideal dua lapisan


In [30]:

jarak = np.array([5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60],
                 dtype=float)
v1_model = 500.0   # m/s, parameter simulasi
v2_model = 1500.0  # m/s, parameter simulasi
ti_model = 0.040   # s, waktu intersep simulasi




# Waktu tempuh model: gelombang langsung dan refraksi



In [31]:

waktu = np.where(
    jarak <= 30,
    jarak / v1_model,
    ti_model + jarak / v2_model
)

data = pd.DataFrame({
    "Jarak_m": jarak,
    "Waktu_Tiba_s": waktu
})



# Pemisahan cabang berdasarkan asumsi data simulasi


In [32]:

langsung = data["Jarak_m"] <= 25
refraksi = data["Jarak_m"] >= 35



# Regresi waktu terhadap jarak: t = m*x + b


In [33]:

m1, b1 = np.polyfit(
    data.loc[langsung, "Jarak_m"],
    data.loc[langsung, "Waktu_Tiba_s"], 1
)
m2, b2 = np.polyfit(
    data.loc[refraksi, "Jarak_m"],
    data.loc[refraksi, "Waktu_Tiba_s"], 1
)

if m1 <= 0 or m2 <= 0:
    raise ValueError("Gradien harus positif; periksa data dan cabang.")

v1 = 1 / m1
v2 = 1 / m2
ti = b2

if v2 <= v1:
    raise ValueError(
        "Model refraksi dua lapisan memerlukan v2 > v1."
    )



# Kedalaman batas lapisan untuk model dua lapisan horizontal


In [34]:

kedalaman = ti * v1 * v2 / (2 * np.sqrt(v2**2 - v1**2))

data["Cabang_Asumsi"] = np.where(
    data["Jarak_m"] <= 25, "Langsung",
    np.where(data["Jarak_m"] >= 35, "Refraksi", "Transisi")
)
data.to_csv(OUTPUT / "tabel_waktu_tempuh.csv", index=False)

print("=" * 58)
print("ANALISIS SEISMIK REFRAKSI")
print("DATA ILUSTRASI/SIMULASI — BUKAN DATA LAPANGAN")
print("=" * 58)
print(data.to_string(index=False, formatters={
    "Waktu_Tiba_s": lambda x: f"{x:.5f}"
}))
print(f"\nGradien cabang langsung : {m1:.8f} s/m")
print(f"Kecepatan lapisan 1     : {v1:.2f} m/s")
print(f"Gradien cabang refraksi : {m2:.8f} s/m")
print(f"Kecepatan lapisan 2     : {v2:.2f} m/s")
print(f"Waktu intersep          : {ti:.5f} s")
print(f"Kedalaman batas model   : {kedalaman:.2f} m")


ANALISIS SEISMIK REFRAKSI
DATA ILUSTRASI/SIMULASI — BUKAN DATA LAPANGAN
 Jarak_m Waktu_Tiba_s Cabang_Asumsi
     5.0      0.01000      Langsung
    10.0      0.02000      Langsung
    15.0      0.03000      Langsung
    20.0      0.04000      Langsung
    25.0      0.05000      Langsung
    30.0      0.06000      Transisi
    35.0      0.06333      Refraksi
    40.0      0.06667      Refraksi
    45.0      0.07000      Refraksi
    50.0      0.07333      Refraksi
    55.0      0.07667      Refraksi
    60.0      0.08000      Refraksi

Gradien cabang langsung : 0.00200000 s/m
Kecepatan lapisan 1     : 500.00 m/s
Gradien cabang refraksi : 0.00066667 s/m
Kecepatan lapisan 2     : 1500.00 m/s
Waktu intersep          : 0.04000 s
Kedalaman batas model   : 10.61 m



# Grafik travel time


In [35]:

plt.figure(figsize=(9, 6))
plt.scatter(
    data.loc[langsung, "Jarak_m"],
    data.loc[langsung, "Waktu_Tiba_s"],
    label="Cabang langsung (asumsi)"
)
plt.scatter(
    data.loc[refraksi, "Jarak_m"],
    data.loc[refraksi, "Waktu_Tiba_s"],
    label="Cabang refraksi (asumsi)"
)

x1 = np.linspace(
    data.loc[langsung, "Jarak_m"].min(),
    data.loc[langsung, "Jarak_m"].max(), 100
)
x2 = np.linspace(
    data.loc[refraksi, "Jarak_m"].min(),
    data.loc[refraksi, "Jarak_m"].max(), 100
)

plt.plot(x1, m1*x1+b1, label=f"Regresi langsung: v={v1:.0f} m/s")
plt.plot(x2, m2*x2+b2, label=f"Regresi refraksi: v={v2:.0f} m/s")
plt.xlabel("Jarak sumber–geofon (m)")
plt.ylabel("Waktu tiba (s)")
plt.title("Grafik Travel Time Seismik Refraksi — Data Simulasi")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "grafik_travel_time.png", dpi=300)
plt.close()




## Hasil


In [36]:

- Kecepatan lapisan pertama: {v1:.2f} m/s
- Kecepatan lapisan kedua: {v2:.2f} m/s
- Waktu intersep: {ti:.5f} s
- Estimasi kedalaman batas model: {kedalaman:.2f} m


SyntaxError: invalid decimal literal (3753100084.py, line 1)


## Pembahasan
Data ilustrasi memperlihatkan dua cabang waktu tempuh. Kebalikan gradien
regresi menghasilkan estimasi kecepatan masing-masing cabang. Dengan
asumsi model dua lapisan horizontal dan kecepatan lapisan kedua lebih
tinggi, waktu intersep digunakan untuk memperkirakan kedalaman batas.

Hasil ini konsisten dengan parameter pembentuk data simulasi dan tidak
menunjukkan kondisi lokasi nyata. Data lapangan dan verifikasi independen
diperlukan sebelum interpretasi geofisika dapat dipertanggungjawabkan.

## Kesimpulan
Program mendemonstrasikan tabel waktu tempuh, regresi linear, estimasi
kecepatan, kedalaman model, dan grafik travel time. Semua angka berasal
dari data simulasi.
"""


In [ ]:

print(f"\nHasil disimpan di: {()}")
print("- tabel_waktu_tempuh.csv")
print("- grafik_travel_time.png")
print("- ringkasan_analisis.md")



Hasil disimpan di: ()
- tabel_waktu_tempuh.csv
- grafik_travel_time.png
- ringkasan_analisis.md
